<a href="https://colab.research.google.com/github/Murcha1990/ML_AI24/blob/main/Hometasks/Base/AI_HW6_uplift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> Задание по Uplift-моделированию </h1>

<h2>Введение</h2>

Перед вами типичная задача, возникающая при работе с моделями кампейнинга в банке: заказчик запустил несколько пилотов по взаимодействию с клиентами с помощью разных каналов: push в мобильном приложении, sms, баннеры в мобильном приложении и реклама в других приложениях экосистемы. Заказчик хотел бы понимать, какой канал взаимодействия с клиентом наиболее эффективен для каждого клиента из клиентской базы. Кампании планируются и запускаются в ежемесячном режиме. Иными словами, заказчик хотел бы в идеале ежемесячно получать список клиентов, которым необходимо отправить коммуникацию с указанием канала и прироста вероятности покупки в случае, если клиенту отправят коммуникацию по сравнению с тем случаем, когда клиенту коммуникацию не отправят.

<b>Таким образом: </b>
1.	У нас есть база клиентов (клиенты, имеющие id в банке). По данной базе осуществляется рассылка тех или иных стимулирующих коммуникаций по различным продуктам, каналам (например SMS, Push, баннеры в мобильном приложении и т.д.) и сегментам клиентов
2.	Признаковое описание клиента состоит из различных агрегатов действий клиента за месяц или его объективных характеристик: например, средняя сумма средств на депозитах за месяц, среднее число кликов клиента в день за месяц в разделе "инвестиции" в мобильном приложении или возраст клиента
3.	При формировании обучающей/тестовой выборки допускается, что один и тот же клиент за разные месяцы — это разные объекты. То есть допускается, что клиент в феврале и клиент в марте — это разные клиенты (то есть мы можем оперировать с ними как с разными сущностями).
4.	Агрегаты действий клиента за месяц появляются примерно 10 числа следующего месяца. То есть, например, агрегаты за декабрь появляются 10 января. В свою очередь списки клиентов, которым необходимо осуществить рассылку должны быть сформированы ориентировочно 20 числа предыдущего месяца. Таким образом, <b> модель должна быть обучена делать предсказания с лагом в два месяца </b>, то есть должна делать предсказание на март по клиентским агрегатам за январь. Обязательно учтите это при обучении модели (в противном случае можно получить лик таргета, так как часто величину, которую мы предсказываем уже есть в клиентских агрегатах, но смещенная на два месяца).


## Оценивание задания:

Всего за задание можно получить 50 первичных баллов, которые затем переводятся в 10-балльную шкалу делением не 5.

Скачаем архив с данными по ссылке и разархивируем.

In [2]:
!pip3 install gdown -q


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip


In [3]:
import gdown

url = 'https://drive.google.com/uc?id=19nKGaxm3RwHxh2UWPo537_-MDx21AkHO'
output = 'Data.zip'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=19nKGaxm3RwHxh2UWPo537_-MDx21AkHO
From (redirected): https://drive.google.com/uc?id=19nKGaxm3RwHxh2UWPo537_-MDx21AkHO&confirm=t&uuid=d7075807-40a6-4f66-b0f5-b7052864750c
To: /Users/luka.markov/git/luka/ML_AI24_course/Hometasks/Base/Data.zip
100%|██████████| 289M/289M [03:24<00:00, 1.41MB/s] 


'Data.zip'

In [1]:
import zipfile

with zipfile.ZipFile('Data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

OSError: [Errno 30] Read-only file system: '/content'

<h2>Описание данных</h2>

Перед вами несколько наборов данных, на основе которых вам будет необходимо обучить Uplift модели, сделать прогноз на нужный месяц и решить, кому из клиентов отправлять коммуникацию, а кому коммуникацию отправлять не следует.

<h3>Features </h3> Признаки клиентов, клиентские агрегаты, которые описывают поведение клиентов <br>

AGGS_FINAL
1. user_id - id клиента
2. report_dt - месяц, на который актуальны признаки
3. city - город, в котором живет клиент
4. age - возраст клиента
5. x1 – x9 - числовые признаки клиента, характеризующие поведение клиента

Первичный ключ таблицы - user_id + report_dt

<h3> Contracts </h3> Таблица с покупками продуктов.


CONTRACTS_FINAL
1. contract_id - id покупки
2. user_id - id пользователя, который совершил покупку
3. product_id - id продукта, который был куплен
4. contract_ts – дата момента, когда была совершена покупка

Первичный ключ - contract_id


<h3> Campaings </h3> Кампании, которые проводились (под кампанией мы понимаем рассылку sms, push и т.д).

CAMPAINGS

1. campaing_id - id кампании, первичный ключ таблицы
2. product_id - продукт, по которому проводилась кампания (считаем, что продукты не конкурируют друг с другом)
3. channel - канал, в котором проводилась кампания


<h3> People_in_campaings </h3> Люди, которые принимали участие в кампаниях.

PEOOPLE_IN_CAMPAINGS

1. campaing_id - id кампании
2. user_id - id пользователя, который попал в кампанию
3. флаг целевой (1) и контрольной (0) группы (целевая группа - это те, кто получил коммуникацию, а контрольная - те, кто нет)
4. delivery_ts - timestamp, когда клиенту фактически была доставлена коммуникация (для контрольной группы nan, подумайте почему)

Первичный ключ данной таблицы - user_id + campaing_id


<h1> Постановка задачи </h1> В ноябре 2024 проводилось несколько кампаний по продукту с id 0001 (фактически клиенту рассылалось одно и тоже сообщение, но в разных каналах). Вам необходимо по данным кампаниям построить модель, которая будет определять лучший канал коммуникации каждого клиента и определить, кому из клиентов в марте 2025 отправить какую коммуникацию, а кому коммуникацию вообще отправлять не следует.
Ответ нужно представить в следующем виде (report_dt – дата фичей):

<table>
  <thead>
    <tr>
      <th>user_id</th>
      <th>report_dt</th>
      <th>channel</th>
      <th>uplift</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>10045</td>
      <td>2025-01-31</td>
      <td>banner</td>
      <td>0.07</td>
    </tr>
    <tr>
      <td>10046</td>
      <td>2025-01-31</td>
      <td>no_comm</td>
      <td>0.00</td>
    </tr>
    <tr>
      <td>10047</td>
      <td>2025-01-31</td>
      <td>sms</td>
      <td>0.23</td>
    </tr>
    <tr>
      <td>10048</td>
      <td>2025-01-31</td>
      <td>push</td>
      <td>0.19</td>
    </tr>
  </tbody>
</table>

<h1> Декомпозиция задачи </h1>

<h2> 1.	Сбор и анализ таргета (18 баллов)</h2>

Прежде всего, вам необходимо собрать целевое событие, которое вы собираетесь прогнозировать. В данном случае целевое событие - это покупка продукта 0001 пользователем, участвовавшем в кампании. Обратите внимание, что не все пользователи получают коммуникацию одновременно (delivery_ts в таблице People_in_campaings). Согласно правилу, согласованному с заказчиком, <b> человек из целевой группы купил продукт после коммуникации - это значит, что он купил его в течение 2х недель после получения сообщения, а человек из контрольной - в течение 3х недель с момента старта кампании (старт кампании - начало месяца). </b> То есть для определенной кампании, для каждого клиента, попавшего в кампанию, вам надо будет найти его покупки данного продукта, а потом основываяся на данном правиле превратить покупки в 0 или 1. <br> На выходе у вас должен появиться таблица с целевым действием для каждого канала (колонки client_id, report_dt,  target), где таргет - это бинарная переменная (0 или 1). Колонка report_dt вам нужна как техническая колонка для дальнейших джоинов.<br><br>

Проведите анализ полученных данных (до присоединения клиентских агрегатов). Какие проблемы и сложности в данных вы обнаружили? Что с ними можно сделать? Какая из кампаний наиболее эффективная? Подготовьте выводы по полученным инсайтам.


**Комментарий по заданиям и оцениванию:**

* Вы должны самостоятельно сделать join нескольких таблиц, самостоятельно собрать целевое действие

* Представлены 4 различных канала, за таргет по каждому из каналов можно получить **максимум 2 балла**:
    * 1 балл за то, что просчитано целевое действие для целевой группы (покупка в
течение одной-двух недель с момента получения коммуникации)
    * 1 балл за то, что просчитано целевое действие для контрольной группы (покупка в течение двух-трех недель с момента старта кампании) и сделана таблица в требуемом формате

* Обратите внимание, что не во всех кампаниях содержатся корректные данные для проведения моделирования, и вам необходимо провести анализ данных и в случае выявленных некорректностей - описать их, и не проводить моделирование для "сломанной" кампании  
    * За данный анализ можно получить **8 баллов**

* Вы должны оценить эффективность кампаний по uplift (cреднее значение таргета в целевой минус среднее значение таргета в контрольной группе)
    * За данный анализ можно получить **2 балла**

In [2]:
import pandas as pd

In [20]:
aggs_df = pd.read_csv('content/AGGS_FINAL.csv', parse_dates=['report_dt'])
aggs_df = aggs_df.drop('Unnamed: 0', axis=1)

In [49]:
aggs_df.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,report_dt,user_id,age,city,report_dt_YM
0,0.654343,-1.439286,-0.011475,2.039457,0.843580,-0.977480,-0.768019,-1.044127,0.025673,2025-01-31,1066338,26,Ufa,2025-01
1,2.583579,1.755569,3.360186,-1.122864,0.034201,-0.269607,-1.503646,1.040289,-1.691606,2024-11-30,13900,35,Ufa,2024-11
2,0.296030,-0.937075,1.073280,1.874636,-0.981216,-1.100187,-0.331181,-1.575637,0.474965,2025-03-31,4063636,28,Ufa,2025-03
3,2.329328,-1.345159,0.345066,0.755373,-0.082842,0.028439,0.919211,0.808793,-0.560004,2025-03-31,1025488,27,Moscow,2025-03
4,0.167643,1.587099,0.165357,0.289758,-1.108840,-1.501819,0.615588,1.631203,-0.208419,2025-02-28,4040555,37,Moscow,2025-02


x1           0
x2           0
x3           0
x4           0
x5           0
x6           0
x7           0
x8           0
x9           0
report_dt    0
user_id      0
age          0
city         0
dtype: int64

In [ ]:
aggs_df['city'] = aggs_df['city'].astype(str)

In [44]:
aggs_df.dtypes

x1                  float64
x2                  float64
x3                  float64
x4                  float64
x5                  float64
x6                  float64
x7                  float64
x8                  float64
x9                  float64
report_dt    datetime64[ns]
user_id               int64
age                   int64
city                 object
dtype: object

In [45]:
aggs_df['report_dt_YM'] = aggs_df['report_dt'].dt.to_period('M')

In [46]:
aggs_df.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,report_dt,user_id,age,city,report_dt_YM
0,0.654343,-1.439286,-0.011475,2.039457,0.843580,-0.977480,-0.768019,-1.044127,0.025673,2025-01-31,1066338,26,Ufa,2025-01
1,2.583579,1.755569,3.360186,-1.122864,0.034201,-0.269607,-1.503646,1.040289,-1.691606,2024-11-30,13900,35,Ufa,2024-11
2,0.296030,-0.937075,1.073280,1.874636,-0.981216,-1.100187,-0.331181,-1.575637,0.474965,2025-03-31,4063636,28,Ufa,2025-03
3,2.329328,-1.345159,0.345066,0.755373,-0.082842,0.028439,0.919211,0.808793,-0.560004,2025-03-31,1025488,27,Moscow,2025-03
4,0.167643,1.587099,0.165357,0.289758,-1.108840,-1.501819,0.615588,1.631203,-0.208419,2025-02-28,4040555,37,Moscow,2025-02


In [11]:
campaings_df = pd.read_csv('content/CAMPAINGS.csv')
campaings_df = campaings_df.drop('Unnamed: 0', axis=1)

In [12]:
campaings_df.head()

,campaing_id,product_id,channel
0,iddqd,1,push
1,idclip,1,sms
2,iddt,1,banner
3,idkfa,1,other_ads


In [18]:
campaings_df['product_id'].nunique()

1

In [32]:
campaings_df = campaings_df.drop('product_id', axis=1)

In [50]:
contracts_df = pd.read_csv('content/CONTRACTS_FINAL.csv', parse_dates=['contract_date'])
contracts_df = contracts_df.drop('Unnamed: 0', axis=1)
contracts_df = contracts_df.drop('product_id', axis=1)

In [51]:
contracts_df.head()

,user_id,contract_date,contract_id
0,4008279,2024-11-03,0001_2024-11-03_4008279
1,2079035,2024-11-08,0001_2024-11-08_2079035
2,103088,2024-11-13,0001_2024-11-13_103088
3,2026788,2024-11-02,0001_2024-11-02_2026788
4,52269,2024-11-17,0001_2024-11-17_52269


In [52]:
contracts_df.dtypes

user_id                   int64
contract_date    datetime64[ns]
contract_id              object
dtype: object

In [53]:
contracts_df['contract_date_YM'] = contracts_df['contract_date'].dt.to_period('M')

In [54]:
contracts_df.head()

,user_id,contract_date,contract_id,contract_date_YM
0,4008279,2024-11-03,0001_2024-11-03_4008279,2024-11
1,2079035,2024-11-08,0001_2024-11-08_2079035,2024-11
2,103088,2024-11-13,0001_2024-11-13_103088,2024-11
3,2026788,2024-11-02,0001_2024-11-02_2026788,2024-11
4,52269,2024-11-17,0001_2024-11-17_52269,2024-11


In [55]:
contracts_with_users = pd.merge(contracts_df, aggs_df, left_on=['user_id', 'contract_date_YM'], right_on=['user_id', 'report_dt_YM'], how='inner')

In [56]:
contracts_with_users.head()

,user_id,contract_date,contract_id,contract_date_YM,x1,x2,x3,x4,x5,x6,x7,x8,x9,report_dt,age,city,report_dt_YM
0,4008279,2024-11-03,0001_2024-11-03_4008279,2024-11,2.525733,0.688233,1.266474,0.696667,-0.093837,0.068828,0.217220,1.557408,-1.192181,2024-11-30,45,Smolensk,2024-11
1,2079035,2024-11-08,0001_2024-11-08_2079035,2024-11,2.932306,0.491722,2.220046,0.223172,0.202512,0.614390,-1.070296,1.270988,-1.842829,2024-11-30,43,Smolensk,2024-11
2,103088,2024-11-13,0001_2024-11-13_103088,2024-11,1.330401,1.585000,2.246719,0.789387,0.256699,0.262553,-3.460479,-1.262026,-2.406468,2024-11-30,21,Moscow,2024-11
3,2026788,2024-11-02,0001_2024-11-02_2026788,2024-11,0.304396,1.197201,0.895842,3.583583,-0.201092,0.379481,-1.305104,1.025323,0.571717,2024-11-30,29,Smolensk,2024-11
4,52269,2024-11-17,0001_2024-11-17_52269,2024-11,2.285360,-0.525870,-0.446459,0.593999,2.831694,1.261054,0.918734,-1.122946,-3.241353,2024-11-30,31,Smolensk,2024-11


In [29]:
people_in_campaings_df = pd.read_csv('content/PEOPLE_IN_CAMPAINGS_FINAL.csv')
people_in_campaings_df = people_in_campaings_df.drop('Unnamed: 0', axis=1)
people_in_campaings_df['delivery_date'] = pd.to_datetime(people_in_campaings_df['delivery_date'], errors='coerce')

In [30]:
people_in_campaings_df.head()

,campaing_id,user_id,t_flag,delivery_date
0,idclip,1099975,1,2024-11-06
1,iddqd,1162,1,2024-11-08
2,iddqd,42991,1,2024-11-07
3,idclip,142343,0,NaT
4,iddqd,24623,0,NaT


In [31]:
people_in_campaings_df.dtypes

campaing_id              object
user_id                   int64
t_flag                    int64
delivery_date    datetime64[ns]
dtype: object

In [39]:
final_df = pd.merge(people_in_campaings_df, campaings_df, on='campaing_id', how='inner')

In [40]:
final_df.head()

,campaing_id,user_id,t_flag,delivery_date,channel
0,idclip,1099975,1,2024-11-06,sms
1,iddqd,1162,1,2024-11-08,push
2,iddqd,42991,1,2024-11-07,push
3,idclip,142343,0,NaT,sms
4,iddqd,24623,0,NaT,push


In [47]:
final_df['delivery_date_YM'] = final_df['delivery_date'].dt.to_period('M')

In [48]:
final_df.head()

,campaing_id,user_id,t_flag,delivery_date,channel,delivery_date_YM
0,idclip,1099975,1,2024-11-06,sms,2024-11
1,iddqd,1162,1,2024-11-08,push,2024-11
2,iddqd,42991,1,2024-11-07,push,2024-11
3,idclip,142343,0,NaT,sms,NaT
4,iddqd,24623,0,NaT,push,NaT


### ваши выводы здесь

<h2> 2. Клиентские агрегаты (12 баллов)</h2>

Присоедините клиентские агрегаты (будьте внимательны, присоедините агрегаты за корректный месяц) и изучите полученные данные.

**Комментарий по заданиям и оцениванию:**

* Вы должны корректно присоединить клиентские агрегаты со смещением на два месяца, чтобы не было лика таргета. За данное действие можно получить **4 балла**

* Далее вы должен сделать UPLIFT EDA, которые обсуждались на лекции и показывались в практических ноутбуках. В ходе анализа вы должны проверить корректность данных по рекламным кампаниям и решить, что делать со "сломанными" кампаниями. По итогам анализа подготовьте выводы. За данное действие можно получить **8 баллов**

In [ ]:
# ваш код здесь

### ваши выводы здесь

<h2> 3. Построение моделей и оценка их качества (14 баллов)</h2>

Постройте Uplift модели по собранным кампаниям, проведите тюнинг гиперпараметров и оцените их качество (qini score). Для каждой модели также постройте qini-curve.

**Комментарий по заданиям и оцениванию:**

* Реализован только подход Solomodel без дополнительных библиотек и калибровок  - **1 балл**

* Реализован Solomodel или Twomodel через Sklift или CausalML - **2 балла**

* Учтена калибровка Metalearner'ах - **2 балла**

* Корректно реализован ClassTransformation - **2 балла**

* Реализован UpliftRandomForest - **4 балла**

* Использованы пайплайны в Sklift - **2 балла**

* Реализован тюнинг ( Gridsearch \ Optuna ) - **1 балл**

In [ ]:
# ваш код здесь

<h2>4. Подготовка ответа в требуемом формате и подготовка выводов (6 баллов)</h2>

a) Сделайте скоринг нужных клиентов, подготовьте ответ в требуемом формате

б) Сделайте краткую аналитику того, какой канал взаимодействия наиболее предпочтителен

в) Сделайте выводы по проделанной работе

**Комментарий по заданиям и оцениванию:**

* Подготовлен только ответ - **1 балл**
* Подготовлен содержательный вывод по проделанной работе - **4 балла**
* Корректно принято решение об отправке/не отправке коммуникации клиентам в зависимости от значений Uplift - **1 балл**

In [ ]:
# ваш код здесь

### ваши выводы здесь